In [ ]:
import sys, pandas, sklearn
print(sys.executable)
print(pandas.__version__, sklearn.__version__)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_excel('../data/Assignment_Train.xlsx', na_values=["NaT"], keep_default_na=True)
df

In [ ]:
df.columns

## Target EDA

In [ ]:
print(df["Target"].dtype)
print(df["Target"].value_counts(dropna=False))
print(df["Target"].value_counts(normalize=True, dropna=False).round(3))

In [ ]:
is_text_nat = df["Target"].astype(str) == "NaT"
print(is_text_nat.sum())
df.loc[is_text_nat, ["UID", "Target"]]

In [ ]:
train_df = df.copy()
target_num = pd.to_numeric(train_df["Target"], errors="coerce")
non_finite = target_num.notna() & ~np.isfinite(target_num)

print("Raw target values, including missing:", train_df["Target"].value_counts(dropna=False))
print("\nValues that could not be parsed as numbers:")
print(train_df.loc[target_num.isna(), "Target"].value_counts(dropna=False))
print("\nNon-finite numeric values:")
print(train_df.loc[non_finite, "Target"].value_counts(dropna=False))
print("\nParsed values other than 0 or 1:")
print(target_num.loc[target_num.notna() & ~target_num.isin([0, 1])].value_counts())

In [ ]:
keep = target_num.notna() & np.isfinite(target_num)
assert target_num.loc[keep].isin([0, 1]).all(), "Target contains unexpected values"

train = train_df.loc[keep].copy()
train["Target"] = target_num.loc[keep].astype("int8")

print("Rows excluded for missing/non-finite target:", int((~keep).sum()))
print(train["Target"].value_counts())
print("Positive rate:", train["Target"].mean())

In [ ]:
# Class imbalance

counts = train["Target"].value_counts().sort_index()
shares = counts / counts.sum()

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["0: not delinquent", "1: delinquent"], counts.values, color=["blue", "red"], width=0.5)
ax.bar_label(bars, labels=[f"{n:,}\n({p:.1%})" for n, p in zip(counts, shares)], padding=3)
ax.set_ylabel("Borrowers")
ax.set_title("Target class balance (training set)")
ax.set_ylim(0, counts.max() * 1.2)  
ax.spines[["top", "right"]].set_visible(False)
plt.show()

Given dataset has a heavy class imbalance towards the non-delinquent which is why the accuracy metric is wrong to identify the model's performance and also model is more likely to develop a bias towards the non-delinquent.

To be found, whether the sampling technique improve the AUC vs no sampling

In [ ]:
n_bins = 20
uid_bin = pd.qcut(train["UID"], q=n_bins, labels=False)   # 20 groups of ~500 rows, in UID order
rate = train.groupby(uid_bin)["Target"].mean()
overall = train["Target"].mean()

# How much would a ~500-row group wobble purely by chance? (binomial standard error)
group_size = len(train) / n_bins
se = (overall * (1 - overall) / group_size) ** 0.5

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.fill_between(rate.index, overall - 1.96 * se, overall + 1.96 * se,
                color="grey", alpha=0.15, label="Expected range from chance (95%)")
ax.axhline(overall, color="grey", linestyle="--", linewidth=1, label=f"Overall rate {overall:.1%}")
ax.plot(rate.index, rate.values, marker="o", linewidth=2, color="#4C72B0", label="Rate in group")
ax.set_xlabel(f"UID group (1 to {n_bins}, in file order, ~{group_size:.0f} rows each)")
ax.set_ylabel("Delinquency rate")
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_title("Delinquency rate across UID order")
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

## Feature EDA

In [ ]:
feature_cols = [c for c in train.columns if c not in ("UID", "Target")]
print(len(feature_cols))
print(train[feature_cols].dtypes.value_counts())

49 features have mixed type of entries (Before na_values=["NaT])

In [ ]:
obj_cols = train[feature_cols].select_dtypes(include="object").columns.tolist()

rows = []
for col in obj_cols:
    type_counts = train[col].map(lambda v: type(v).__name__).value_counts()
    rows.append({"column": col,
                 "types": dict(type_counts),
                 "n_unique": train[col].nunique(dropna=False)})

type_report = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
type_report

Most of the 49 feature columns have NaT which is missing value in most features except where NaT mean not applicable or not checked.
To be decided whether these feature provide any value to the prediction or not

NaT is the dataset wide missing placeholder which needs to treated

After using na_values in pandas the distribution of features changed from 49 object types to just 1 object types

Before the target vs feature analysis, it is better to split the training set into the train and val so that val is never touched and the analysis comes only from the actual train set and the val set truly remains unseen

70:30 split reasoning - Gives more positive holdouts which in turn would provide model to learn about the dilenquent class more and give a more stable estimate

In [ ]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42   
dev, holdout = train_test_split(
    train,
    test_size=0.30,              
    stratify=train["Target"],    
    random_state=RANDOM_STATE,
)

print("dev:    ", dev.shape,     "positive rate:", round(dev["Target"].mean(), 4))
print("holdout:", holdout.shape, "positive rate:", round(holdout["Target"].mean(), 4))

assert set(dev["UID"]).isdisjoint(holdout["UID"])
assert len(dev) + len(holdout) == len(train)

In [ ]:
feature_cols = [c for c in train.columns if c not in ("UID", "Target")]

profile = pd.DataFrame({
    "dtype":        train[feature_cols].dtypes.astype(str),
    "pct_missing":  (train[feature_cols].isna().mean() * 100).round(1),
    "n_unique":     train[feature_cols].nunique(),            # excludes NaN
    "top_share":    train[feature_cols].apply(
                        lambda s: s.value_counts(normalize=True).iloc[0] if s.notna().any() else np.nan
                    ).mul(100).round(1),
})

numeric = train[feature_cols].select_dtypes("number")
profile = profile.join(numeric.agg(["min", "median", "max"]).T)

pd.set_option("display.max_rows", 100)
profile.sort_values("n_unique", ascending=False)

1. Email Attributes.domainUpdatedDate, Email Social Advance.paytm, Phone Social Advance.paytm, Phone Name Attributes.revocationMonth, Email Social Advance.skype, Phone Social Advance.skype - ompletely empty feature or useless features
2. On sorting on uniqueness - 
    name	
    Phone to Name.UPIname	
    Phone Name Attributes.digitalage	
    Email Name Attributes.digitalage	
    Email Attributes.digitalAge	int64	
    Email Name Attributes.phoneNumber	
    Phone Name Attributes.phoneNameDigitalAge	
    Email Name Attributes.emailNameDigitalAge
these features have the most unique entires but other than these features there are hardly any unique entries in any other features
Also for the features having unique value as 1, are basically useless since they have just 1 value

In [ ]:
median_minus1 = profile.index[profile["median"].eq(-1)]

sentinel_summary = pd.DataFrame({
    "median": profile.loc[median_minus1, "median"],
    "n_minus1": train[median_minus1].eq(-1).sum(),
    "pct_minus1": (train[median_minus1].eq(-1).mean() * 100).round(1),
    "pct_missing": profile.loc[median_minus1, "pct_missing"],
}).sort_values("pct_minus1", ascending=False)

display(sentinel_summary)

These six features are effectively 59–84% missing, coded as −1, even though pct_missing shows 0%.

I assume −1 means 'not found / no data' across all these fields. It's documented for Email Attributes.digitalAge, it's impossible as a real value for ages and 0–100 scores, and it co-occurs across fields from the same lookup. Risk: if −1 meant something else in some field, my handling would be wrong for that field.

In [ ]:
col = "Phone Name Attributes.digitalage"
print((train[col] > 10_950).sum(), "rows above ~30 years")
train[col].sort_values(ascending=False).head(10)

The one outlier in the phone name attributes digital age

In [ ]:
cat_cols = train[feature_cols].select_dtypes(include=["str", "object"]).columns
cat_cols = [c for c in cat_cols if train[c].nunique() < 30]  

for c in cat_cols:
    print(f"\n{c}   (missing {train[c].isna().mean():.1%})")
    print(train[c].value_counts(dropna=False).head(10).to_string())

In [ ]:
pd.crosstab(train["Email Attributes.footprint"],
            train["Email Attributes.emailFinalRecommendation"])

In [ ]:
train[train.duplicated(subset=train.columns.drop(["UID", "Target"]), keep=False)]

In [ ]:
train.duplicated(subset=train.columns.drop(["UID", "Target"]), keep=False).sum()

In [ ]:
train.duplicated(subset=feature_cols, keep=False).sum()

In [ ]:
has_phone = train["Email Name Attributes.phoneNumber"] != -1
train[has_phone].duplicated(subset=["Email Name Attributes.phoneNumber"], keep=False).sum()

print(has_phone.sum(), "rows with phone number")
print(train[has_phone].duplicated(subset=["Email Name Attributes.phoneNumber"], keep=False).sum(), "duplicate rows with phone number")

In [ ]:
phone_col = "Email Name Attributes.phoneNumber"
with_phone = train.loc[has_phone]
dup_phone = with_phone.duplicated(subset=[phone_col], keep=False)
print(dup_phone.sum(), "rows share a phone number with another row")
with_phone.loc[dup_phone, ["UID", "name", phone_col, "Target"]].sort_values(phone_col)

In [ ]:
upi_col = "Phone to Name.UPIname"
has_upi = train[upi_col].notna()

person_key = pd.DataFrame({
    "name_key": train["name"].str.lower().str.strip(),
    "upi_key":  train[upi_col].astype(str).str.lower().str.strip(),
}).loc[has_upi]                      

dup_person = person_key.duplicated(keep=False)   

print(has_upi.sum(), "rows with a UPI name")
print(dup_person.sum(), "rows share name + UPI name with another row")

dup_idx = person_key.index[dup_person]
train.loc[dup_idx, ["UID", "name", upi_col, "Target"]].sort_values("name")

In [ ]:
def rate_table(data, col, target="Target"):
    """Delinquency rate for each value of `col`; missing values form their own group."""
    groups = data[col].astype("object").where(data[col].notna(), "MISSING")
    out = data.groupby(groups)[target].agg(n="size", delinquent="sum", rate="mean")

    se = np.sqrt(out["rate"] * (1 - out["rate"]) / out["n"])
    out["ci_low"]  = (out["rate"] - 1.96 * se).clip(lower=0)
    out["ci_high"] = (out["rate"] + 1.96 * se).clip(upper=1)

    out["lift"] = out["rate"] / data[target].mean()
    return out.round(3)

rate_table(dev, "Phone Social Advance.whatsapp")

Borrowers without WhatsApp were delinquent 33% of the time, against 7% for those with WhatsApp (433 vs 6,158 people).
Having no WhatsApp is a strong warning sign, possibly because the number isn't someone's main phone or because they're harder to contact.

In [ ]:
# 0/1 features that actually vary (constant columns excluded)
flag_cols = [c for c in feature_cols
             if c in dev.select_dtypes("number").columns
             and set(dev[c].dropna().unique()) == {0, 1}]

rows = []
for col in flag_cols:
    t = rate_table(dev, col)
    rows.append({
        "feature":   col,
        "n_0":       t["n"].get(0.0, 0),        
        "rate_0":    t["rate"].get(0.0),
        "n_1":       t["n"].get(1.0, 0),
        "rate_1":    t["rate"].get(1.0),
        "n_missing": t["n"].get("MISSING", 0),
        "rate_missing": t["rate"].get("MISSING"),
    })

flag_summary = pd.DataFrame(rows).set_index("feature")
flag_summary["gap"] = (flag_summary["rate_0"] - flag_summary["rate_1"]).round(3)
flag_summary.sort_values("gap", ascending=False)

Borrowers usually on the public shopping/social media sites are generally less likely to be delinquents
Combinig multiple public site presence into a single feature which would tell how many platforms does the borrower is (float type).

In [ ]:
def add_platform_share(data):
    """Share of known platforms where the person is registered, separately for email and phone."""
    out = data.copy()
    for source in ["Email", "Phone"]:
        cols = [c for c in data.columns
                if c.startswith(f"{source} Social Advance.")
                and not c.endswith((".paytm", ".skype"))]
        n_known   = data[cols].notna().sum(axis=1)   # platforms with a 0 or 1
        n_present = data[cols].sum(axis=1)           # number of 1s (NaN is skipped)
        out[f"{source.lower()}_platform_known"] = n_known
        out[f"{source.lower()}_platform_share"] = n_present / n_known.replace(0, np.nan)
    return out

dev_fe = add_platform_share(dev)
dev_fe[["email_platform_known", "email_platform_share",
        "phone_platform_known", "phone_platform_share"]].describe().round(2)

In [ ]:
for col in ["email_platform_share", "phone_platform_share"]:
    bins = pd.qcut(dev_fe[col], q=5, duplicates="drop")   # 5 groups of ~1,400 people
    print(f"\n{col}")
    display(rate_table(dev_fe.assign(share_bin=bins), "share_bin"))

Delinquency falls from 13.6% (email) / 15.6% (phone) in the lowest share group to about 6% in the highest, then levels off.
A broader digital presence goes with lower delinquency, but the effect flattens at the top, so the relationship isn't linear.

In [ ]:
def plot_rate(table, title, ax=None):
    """Delinquency rate per group with 95% CI error bars; dashed line = overall dev rate."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 3.5))
    x = range(len(table))
    yerr = [table["rate"] - table["ci_low"], table["ci_high"] - table["rate"]]   # distance below / above the dot
    ax.errorbar(x, table["rate"], yerr=yerr, fmt="o", color="#4C72B0", capsize=4, markersize=7)
    ax.axhline(dev["Target"].mean(), color="grey", linestyle="--", linewidth=1, label="Overall rate")
    ax.set_xticks(list(x))
    ax.set_xticklabels([f"{label}\n(n={n:,})" for label, n in zip(table.index.astype(str), table["n"])], fontsize=8)
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.set_ylabel("Delinquency rate")
    ax.set_title(title)
    ax.legend(frameon=False)
    ax.spines[["top", "right"]].set_visible(False)
    return ax

# Use it for the two platform shares, side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, col in zip(axes, ["email_platform_share", "phone_platform_share"]):
    bins = pd.qcut(dev_fe[col], q=5, labels=["1 (low)", "2", "3", "4", "5 (high)"])
    plot_rate(rate_table(dev_fe.assign(share_bin=bins), "share_bin"), col, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
reliable = flag_summary[flag_summary[["n_0", "n_1"]].min(axis=1) >= 100].sort_values("gap")

fig, ax = plt.subplots(figsize=(8, 9))
y = range(len(reliable))
ax.hlines(y, reliable["rate_1"], reliable["rate_0"], color="lightgrey", linewidth=2)
ax.scatter(reliable["rate_1"], y, color="#4C72B0", s=40, label="On platform (1)", zorder=3)
ax.scatter(reliable["rate_0"], y, color="#C4621D", s=40, label="Not on platform (0)", zorder=3)
ax.axvline(dev["Target"].mean(), color="grey", linestyle="--", linewidth=1, label="Overall rate")
ax.set_yticks(list(y))
ax.set_yticklabels(reliable.index, fontsize=8)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
ax.set_xlabel("Delinquency rate")
ax.set_title("Delinquency rate: on vs not on each platform (groups ≥ 100 people)")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()


In [ ]:
cat_cols_c = [c for c in cat_cols if c != "Phone Name Attributes.revocationMonth"]   # 98.8% missing, dates

for col in cat_cols_c:
    t = rate_table(dev, col)
    plot_rate(t, col)
    plt.show()

Rare categories will have huge error bars. That's the plot telling you not to trust them.

In [ ]:
def numeric_rate_table(data, col, q=5, target="Target"):
    """Delinquency rate by quantile group of valid values; -1 and NaN shown as their own groups."""
    v = data[col]
    sentinel = v.eq(-1)
    valid = v.notna() & ~sentinel

    group = pd.Series("MISSING", index=data.index, dtype="object")
    group[sentinel] = "-1 (not found)"
    group[valid] = "Q" + (pd.qcut(v[valid], q=q, labels=False, duplicates="drop") + 1).astype(str)

    out = data.assign(_g=group, _v=v).groupby("_g").agg(
        n=(target, "size"), rate=(target, "mean"),
        v_min=("_v", "min"), v_max=("_v", "max"))          # value range in each group
    se = np.sqrt(out["rate"] * (1 - out["rate"]) / out["n"])
    out["ci_low"]  = (out["rate"] - 1.96 * se).clip(lower=0)
    out["ci_high"] = (out["rate"] + 1.96 * se).clip(upper=1)
    return out.round(3)

numeric_cols_c = [
    "Email Attributes.digitalAge", "Email Name Attributes.digitalage",
    "Email Name Attributes.emailNameDigitalAge", "Phone Name Attributes.digitalage",
    "Phone Name Attributes.phoneNameDigitalAge", "Email Name Attributes.nameMatchScore",
    "Email Name Attributes.nameEmailMatch", "Email Name Attributes.ndrScore",
    "Email Name Attributes.unrScore", "Email Attributes.uniqueHits",
    "Email GIN.l1Confidence", "Email GIN.l1Count", "Email GIN.l2Confidence", "Email GIN.l2Count",
    "Email GIN.nameDiversity", "Mobile GIN.l1Confidence", "Mobile GIN.l1Count",
    "Mobile GIN.l2Confidence", "Mobile GIN.l2Count", "Mobile GIN.nameDiversity",
]

for col in numeric_cols_c:
    t = numeric_rate_table(dev, col)
    plot_rate(t, col)
    plt.show()

In [ ]:
num_check = [c for c in numeric_cols_c if not c.startswith(("Email GIN", "Mobile GIN"))]
corr = dev[num_check].replace(-1, np.nan).corr(method="spearman")   # -1 excluded: it isn't a real value

# list the most similar pairs
pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))   # upper triangle only (no duplicates)
             .stack().sort_values(key=abs, ascending=False))
pairs.head(10).round(2)

Email Name Attributes.digitalage, Email Name Attributes.emailNameDigitalAge are the most related but digitalage exists for nearly four tiems as many people so keeping that makes more sense since adding to more data.
Also Phone Name Attributes.phoneNameDigitalAge showed a pattern across the data which can be useful so dropping phone digitalage would be good.

In [ ]:
test = pd.read_excel("../data/Assignment_Test.xlsx", na_values=["NaT"])

print("Shape:", test.shape, "| UID unique:", test["UID"].is_unique)
print("Columns match train (minus Target):",
      list(test.columns) == [c for c in train.columns if c != "Target"])

# 1. Missingness and -1 share: train vs test, side by side
compare = pd.DataFrame({
    "train_missing%": train[feature_cols].isna().mean() * 100,
    "test_missing%":  test[feature_cols].isna().mean() * 100,
    "train_minus1%":  train[feature_cols].eq(-1).mean() * 100,
    "test_minus1%":   test[feature_cols].eq(-1).mean() * 100,
}).round(1)
compare["max_diff"] = (compare[["train_missing%", "test_missing%"]].diff(axis=1).iloc[:, 1].abs()
                       .combine(compare[["train_minus1%", "test_minus1%"]].diff(axis=1).iloc[:, 1].abs(), max))
display(compare.sort_values("max_diff", ascending=False).head(10))

# 2. Category levels in test that never appear in train
for col in cat_cols:
    unseen = set(test[col].dropna().unique()) - set(train[col].dropna().unique())
    if unseen:
        print(f"{col}: unseen in train -> {unseen}")

Test data shows lower lookup coverage than train (billing missing 22.8% vs 4.4%; more 'not found' results), and it appears to have been pulled later. Holdout performance may overstate performance on the test population.